#### **Question 1. Sequence Comparison**

For this question, you will work with a one megabase segment of human chromosome 22.

Download, unzip it and extract one megabase segment from 20Mb to 21Mbp:

Q1a. Write a python script (sim_mutations.py) to read in a fasta file, introduce a specified level of mutations, and then output a new fasta file. The input (-i) and output (-o) filenames should be set on the command line as should the level of mutations (-m) and the random seed (-s). For mutations, only introduce substitutions by changing the input base to a different base at random. The substitutions should be inserted at random locations at the specified rate. For example "-m 0.015" should change 1.5% of the original bases.



In [ ]:
#Sim_mutations.py

import argparse
import random as rng

#Need parser to allow for CLI flags
parser = argparse.ArgumentParser(description="sim_mutations argument parser")

parser.add_argument('-i', '--input', type=str, help="path to input file")
parser.add_argument('-o', '--output', type=str, help="path to output file")
parser.add_argument('-m', '--mutation', type=float, help="proportion of input bases to change to different base")
parser.add_argument('-s', '--seed', type=int, help="random seed", default=42)

args = parser.parse_args()

#Open required files
input_file = open(args.input)
output_file = open(args.output, 'w')

#Copy over the fasta header line
output_file.write(f"{input_file.readline().strip()}_mutated\n")

sequence = ''.join(line.strip().upper() for line in input_file.readlines())
length = len(sequence)

sequence = list(sequence)
edits = int(args.mutation * length)

rng.seed(args.seed)
bases_to_edit = rng.sample(range(length), edits)

bases = ['A', 'C', 'G', 'T']
#Replacing each base with some mutated one
for index in bases_to_edit: 
    original = sequence[index] 
    
    # Don't mutate N 
    if original == 'N': 
        continue 
    
    # Choose a base different from the original 
    
    possible_bases = [base for base in bases if base != original]
    sequence[index] = rng.choice(possible_bases) 
    

# Convert back to a string 
sequence = ''.join(sequence)

#Write lines 50 characters per line
line_length = 50
output_file.writelines(f"{sequence[p:p + line_length]}\n" for p in range(0, length, line_length))

#Close files at end
input_file.close()
output_file.close()

Q1b. Now run the script using error rates 0.01, 0.05, 0.10 and 0.15 using three random seeds(e.g. 1, 2, 3), and compare the simulated mutation rate to the average identity (%IDY) computed by mummer. You may find this bash code helpful to evaluate all the runs. Note, with a mutation rate of 0.15 we expect the average identity to by 85%. Some deviation is expected because of rounding, plus mummer may choose to not extend an alignment to the very end of a sequence if there is a cluster of mutations

`make_mutations.sh`:
```
for rate in 0.01 0.05 0.10 0.15
do
    for seed in 1 2 3
    do
        if [ ! -r chr22_mut${rate}_s${seed}.fa ]
        then
            # introduce mutations
            python3 sim_mutations.py -i chr22_orig.fa -o chr22_mut${rate}_s${seed}.fa -m ${rate} -s ${seed}
            nucmer --mum -g 1000 -b 1000 -p chr22_orig_vs_mut${rate}_s${seed} chr22_orig.fa chr22_mut${rate}_s${seed}.fa
            delta-filter -1 chr22_orig_vs_mut${rate}_s${seed}.delta > chr22_orig_vs_mut${rate}_s${seed}.1delta
            show-coords -rcl chr22_orig_vs_mut${rate}_s${seed}.1delta > chr22_orig_vs_mut${rate}_s${seed}.coords
        fi
    done
done

head chr22_orig_vs_mut*.coords
```

============================================================

Answers:

```
==> chr22_orig_vs_mut0.01_s1.coords <==
/Users/ashrithkollu/Documents/Classes/Junior_Fall/CG-ACG/Assignment2/chr22_orig.fa /Users/ashrithkollu/Documents/Classes/Junior_Fall/CG-ACG/Assignment2/chr22_mut0.01_s1.fa
NUCMER

    [S1]     [E1]  |     [S2]     [E2]  |  [LEN 1]  [LEN 2]  |  [% IDY]  |  [LEN R]  [LEN Q]  |  [COV R]  [COV Q]  | [TAGS]
===============================================================================================================================
       1  1000001  |        1  1000001  |  1000001  1000001  |    99.00  |  1000001  1000001  |   100.00   100.00  | chr22:20000000-21000000    chr22:20000000-21000000_mutated

==> chr22_orig_vs_mut0.01_s2.coords <==
/Users/ashrithkollu/Documents/Classes/Junior_Fall/CG-ACG/Assignment2/chr22_orig.fa /Users/ashrithkollu/Documents/Classes/Junior_Fall/CG-ACG/Assignment2/chr22_mut0.01_s2.fa
NUCMER

    [S1]     [E1]  |     [S2]     [E2]  |  [LEN 1]  [LEN 2]  |  [% IDY]  |  [LEN R]  [LEN Q]  |  [COV R]  [COV Q]  | [TAGS]
===============================================================================================================================
       1  1000001  |        1  1000001  |  1000001  1000001  |    99.00  |  1000001  1000001  |   100.00   100.00  | chr22:20000000-21000000    chr22:20000000-21000000_mutated

==> chr22_orig_vs_mut0.01_s3.coords <==
/Users/ashrithkollu/Documents/Classes/Junior_Fall/CG-ACG/Assignment2/chr22_orig.fa /Users/ashrithkollu/Documents/Classes/Junior_Fall/CG-ACG/Assignment2/chr22_mut0.01_s3.fa
NUCMER

    [S1]     [E1]  |     [S2]     [E2]  |  [LEN 1]  [LEN 2]  |  [% IDY]  |  [LEN R]  [LEN Q]  |  [COV R]  [COV Q]  | [TAGS]
===============================================================================================================================
       1  1000001  |        1  1000001  |  1000001  1000001  |    99.00  |  1000001  1000001  |   100.00   100.00  | chr22:20000000-21000000    chr22:20000000-21000000_mutated

==> chr22_orig_vs_mut0.05_s1.coords <==
/Users/ashrithkollu/Documents/Classes/Junior_Fall/CG-ACG/Assignment2/chr22_orig.fa /Users/ashrithkollu/Documents/Classes/Junior_Fall/CG-ACG/Assignment2/chr22_mut0.05_s1.fa
NUCMER

    [S1]     [E1]  |     [S2]     [E2]  |  [LEN 1]  [LEN 2]  |  [% IDY]  |  [LEN R]  [LEN Q]  |  [COV R]  [COV Q]  | [TAGS]
===============================================================================================================================
       1  1000001  |        1  1000001  |  1000001  1000001  |    95.00  |  1000001  1000001  |   100.00   100.00  | chr22:20000000-21000000    chr22:20000000-21000000_mutated

==> chr22_orig_vs_mut0.05_s2.coords <==
/Users/ashrithkollu/Documents/Classes/Junior_Fall/CG-ACG/Assignment2/chr22_orig.fa /Users/ashrithkollu/Documents/Classes/Junior_Fall/CG-ACG/Assignment2/chr22_mut0.05_s2.fa
NUCMER

    [S1]     [E1]  |     [S2]     [E2]  |  [LEN 1]  [LEN 2]  |  [% IDY]  |  [LEN R]  [LEN Q]  |  [COV R]  [COV Q]  | [TAGS]
===============================================================================================================================
       1  1000001  |        1  1000001  |  1000001  1000001  |    95.00  |  1000001  1000001  |   100.00   100.00  | chr22:20000000-21000000    chr22:20000000-21000000_mutated

==> chr22_orig_vs_mut0.05_s3.coords <==
/Users/ashrithkollu/Documents/Classes/Junior_Fall/CG-ACG/Assignment2/chr22_orig.fa /Users/ashrithkollu/Documents/Classes/Junior_Fall/CG-ACG/Assignment2/chr22_mut0.05_s3.fa
NUCMER

    [S1]     [E1]  |     [S2]     [E2]  |  [LEN 1]  [LEN 2]  |  [% IDY]  |  [LEN R]  [LEN Q]  |  [COV R]  [COV Q]  | [TAGS]
===============================================================================================================================
       1  1000001  |        1  1000001  |  1000001  1000001  |    95.00  |  1000001  1000001  |   100.00   100.00  | chr22:20000000-21000000    chr22:20000000-21000000_mutated

==> chr22_orig_vs_mut0.10_s1.coords <==
/Users/ashrithkollu/Documents/Classes/Junior_Fall/CG-ACG/Assignment2/chr22_orig.fa /Users/ashrithkollu/Documents/Classes/Junior_Fall/CG-ACG/Assignment2/chr22_mut0.10_s1.fa
NUCMER

    [S1]     [E1]  |     [S2]     [E2]  |  [LEN 1]  [LEN 2]  |  [% IDY]  |  [LEN R]  [LEN Q]  |  [COV R]  [COV Q]  | [TAGS]
===============================================================================================================================
       1  1000001  |        1  1000001  |  1000001  1000001  |    90.02  |  1000001  1000001  |   100.00   100.00  | chr22:20000000-21000000    chr22:20000000-21000000_mutated

==> chr22_orig_vs_mut0.10_s2.coords <==
/Users/ashrithkollu/Documents/Classes/Junior_Fall/CG-ACG/Assignment2/chr22_orig.fa /Users/ashrithkollu/Documents/Classes/Junior_Fall/CG-ACG/Assignment2/chr22_mut0.10_s2.fa
NUCMER

    [S1]     [E1]  |     [S2]     [E2]  |  [LEN 1]  [LEN 2]  |  [% IDY]  |  [LEN R]  [LEN Q]  |  [COV R]  [COV Q]  | [TAGS]
===============================================================================================================================
       1   999996  |        1   999996  |   999996   999996  |    90.02  |  1000001  1000001  |   100.00   100.00  | chr22:20000000-21000000    chr22:20000000-21000000_mutated

==> chr22_orig_vs_mut0.10_s3.coords <==
/Users/ashrithkollu/Documents/Classes/Junior_Fall/CG-ACG/Assignment2/chr22_orig.fa /Users/ashrithkollu/Documents/Classes/Junior_Fall/CG-ACG/Assignment2/chr22_mut0.10_s3.fa
NUCMER

    [S1]     [E1]  |     [S2]     [E2]  |  [LEN 1]  [LEN 2]  |  [% IDY]  |  [LEN R]  [LEN Q]  |  [COV R]  [COV Q]  | [TAGS]
===============================================================================================================================
       1   999973  |        1   999973  |   999973   999973  |    90.02  |  1000001  1000001  |   100.00   100.00  | chr22:20000000-21000000    chr22:20000000-21000000_mutated

==> chr22_orig_vs_mut0.15_s1.coords <==
/Users/ashrithkollu/Documents/Classes/Junior_Fall/CG-ACG/Assignment2/chr22_orig.fa /Users/ashrithkollu/Documents/Classes/Junior_Fall/CG-ACG/Assignment2/chr22_mut0.15_s1.fa
NUCMER

    [S1]     [E1]  |     [S2]     [E2]  |  [LEN 1]  [LEN 2]  |  [% IDY]  |  [LEN R]  [LEN Q]  |  [COV R]  [COV Q]  | [TAGS]
===============================================================================================================================
       1  1000001  |        1  1000001  |  1000001  1000001  |    85.09  |  1000001  1000001  |   100.00   100.00  | chr22:20000000-21000000    chr22:20000000-21000000_mutated

==> chr22_orig_vs_mut0.15_s2.coords <==
/Users/ashrithkollu/Documents/Classes/Junior_Fall/CG-ACG/Assignment2/chr22_orig.fa /Users/ashrithkollu/Documents/Classes/Junior_Fall/CG-ACG/Assignment2/chr22_mut0.15_s2.fa
NUCMER

    [S1]     [E1]  |     [S2]     [E2]  |  [LEN 1]  [LEN 2]  |  [% IDY]  |  [LEN R]  [LEN Q]  |  [COV R]  [COV Q]  | [TAGS]
===============================================================================================================================
       1   999996  |        1   999996  |   999996   999996  |    85.09  |  1000001  1000001  |   100.00   100.00  | chr22:20000000-21000000    chr22:20000000-21000000_mutated

==> chr22_orig_vs_mut0.15_s3.coords <==
/Users/ashrithkollu/Documents/Classes/Junior_Fall/CG-ACG/Assignment2/chr22_orig.fa /Users/ashrithkollu/Documents/Classes/Junior_Fall/CG-ACG/Assignment2/chr22_mut0.15_s3.fa
NUCMER

    [S1]     [E1]  |     [S2]     [E2]  |  [LEN 1]  [LEN 2]  |  [% IDY]  |  [LEN R]  [LEN Q]  |  [COV R]  [COV Q]  | [TAGS]
===============================================================================================================================
       1   999973  |        1   999973  |   999973   999973  |    85.09  |  1000001  1000001  |   100.00   100.00  | chr22:20000000-21000000    chr22:20000000-21000000_mutated
       ```

The simulated mutation rate is nearly equal to the average identity computed by mummer. The average for 1% is 99% identity, the average for 5% is 95% identity, the average for 10% is 90.02% identity, the average for 15% is 85.09% identity. The mummer identity aligns nearly perfectly with what we expected from the simulations.

Q1c. Now write a python program to compute the Jaccard coefficient between two sequences specified with -a and -b using the specified kmer length (-k) Output the Jaccard coefficient, the ANI (computed by computing the kth root), and the approximate ANI computed using natural log. Convert any lowercase letters to uppercase, and convert any non-ACGT characters to N.

#### **Question 2. de Bruijn Graph construction**

Q2a. Write a script (in python, R, C++, etc) to draw the de Bruijn graph for the following reads using k=3 (assume all reads are from the forward strand, no sequencing errors, complete coverage of the genome).
ATTCA
ATTGA
CATTG
CTTAT
GATTG
TATTT
TCATT
TCTTA
TGATT
TTATT
TTCAT
TTCTT
TTGAT.



In [1]:
text_string = """ATTCA
ATTGA
CATTG
CTTAT
GATTG
TATTT
TCATT
TCTTA
TGATT
TTATT
TTCAT
TTCTT
TTGAT""".split()

print(text_string)
output_file = open("graph.dot", 'w')
output_file.write("strict digraph{\n\n")

k = 3
for kmer in text_string:
    length = len(kmer)

    for i in range(length - k):
        output_file.write(f"    {kmer[i:i+3]} -> {kmer[i+1:i+4]}\n")

output_file.write("}")
output_file.close()

['ATTCA', 'ATTGA', 'CATTG', 'CTTAT', 'GATTG', 'TATTT', 'TCATT', 'TCTTA', 'TGATT', 'TTATT', 'TTCAT', 'TTCTT', 'TTGAT']


<img src = "https://raw.githubusercontent.com/kolluashrith/CG-ACG/refs/heads/main/Assignment2/neato.png" width = 300 />

Q2b. Assume that the maximum number of occurrences of any 3-mer in the actual genome is 4 using the k-mers from Q2a. Write one possible genome sequence. Note this genome sequence may include sequences that are not supported by the underlying reads (this is the well known read decoherency problem for genome assembly).

Answer (possible): TCA CAT ATT TTG TGA GAT ATT TTC TCT CTT TTA TAT ATT TCC TCA CAT ATT TTT

Q2c. What would it take to fully resolve the genome? [In a few sentences]

Answer: Fully resolving the genome would require acquiring information about the exact number of times the kmer shows up in the genome and/or information about how far apart kmers are. Looking at kmer coverage could be a good indicator of the number of times the graph should be passing through the kmer. In addition, using a method like Illumina paired-ends sequencing could allow us to identify the amount of separation between two kmers, allowing us to better predict the actual number of times to go around loops in the graph. These are just two of the pieces we need to put together the genome puzzle since it's necessary to determine how the graph is meant to be linked together.

#### **Question 3: Kmer Analysis**

Use `jellyfish` to count the 21-mers in the reads data (all reads should be counted together). Make sure to use the "-C" flag to count cannonical kmers, otherwise your analysis will not correctly account for the fact that your reads come from either strand of DNA.

Question 3a. How many kmers occur exactly 50 times?

Answer: 932 kmers occur 50 times.

Question 3b. What are the top 10 most frequently occurring kmers?


In [2]:
#Used Claude to somehow get the commands from terminal to work in a notebook. I did it in terminal first.

import glob
import subprocess

subprocess.run(["jellyfish", "count", "-m", "21", "-o", "mer_counts.jf", "-C", "-s", "1000000",
                *glob.glob("./asm/*.fq")], check=True)

subprocess.run(["jellyfish", "merge", "-o", "mer_counts.jf", *glob.glob("mer_counts.jf_*")], check=True)

with open("reads.histo", "w") as fh:
    subprocess.run(["jellyfish", "histo", "mer_counts.jf"], stdout=fh, check=True)

print("Kmer:               Count:")
subprocess.run("jellyfish dump -c mer_counts.jf | sort -k2,2nr | head -n 10", shell=True, check=True)

Kmer:               Count:
AGGTTCAATTCCTGCCGGGCG 93
CGCGCCCGGCAGGAATTGAAC 91
GCGCCCGGCAGGAATTGAACC 91
CAGGTTCAATTCCTGCCGGGC 90
CCCGGCAGGAATTGAACCTGC 90
CCGGCAGGAATTGAACCTGCG 90
GAAGGTCGCAGGTTCAATTCC 90
GCAGGAATTGAACCTGCGACC 90
GCGCGCCCGGCAGGAATTGAA 90
GGCGCGCCCGGCAGGAATTGA 90


CompletedProcess(args='jellyfish dump -c mer_counts.jf | sort -k2,2nr | head -n 10', returncode=0)

Question 3c. What is the estimated genome size based on the kmer frequencies?

Answer: 233,492 bp

Question 3d. How well does the GenomeScope genome size estimate compare to the reference genome?

In [3]:
with open("asm/ref.fa") as file:
    print(sum(len(line.strip()) for line in file if not line.startswith(">")))

233806


Answer: The GenomeScope genome size estimate is very close to the size of the reference genome. It only differs by about 300 or so bases, which is an underestimate by only 0.1%. 


#### **Question 4. De Novo Assembly**

Question 4a. How many contigs were produced?

In [4]:
import subprocess
import os

os.chdir("/Users/ashrithkollu/Documents/Classes/Junior_Fall/CG-ACG/Assignment2/asm/")
subprocess.run([
    "spades.py",
    "--isolate",
    "--pe1-1", "frag180.1.fq",
    "--pe1-2", "frag180.2.fq",
    "--mp1-1", "jump2k.1.fq",
    "--mp1-2", "jump2k.2.fq",
    "-o", "asm",
    "-t", "4",
    "-k", "31",
], check=True)



== Warning ==  output dir is not empty! Please, clean output directory before run.


Command line: /Users/ashrithkollu/Documents/Classes/Junior_Fall/CG-ACG/miniforge3/envs/asn2/bin/spades.py	--isolate	--pe1-1	/Users/ashrithkollu/Documents/Classes/Junior_Fall/CG-ACG/Assignment2/asm/frag180.1.fq	--pe1-2	/Users/ashrithkollu/Documents/Classes/Junior_Fall/CG-ACG/Assignment2/asm/frag180.2.fq	--mp1-1	/Users/ashrithkollu/Documents/Classes/Junior_Fall/CG-ACG/Assignment2/asm/jump2k.1.fq	--mp1-2	/Users/ashrithkollu/Documents/Classes/Junior_Fall/CG-ACG/Assignment2/asm/jump2k.2.fq	-o	/Users/ashrithkollu/Documents/Classes/Junior_Fall/CG-ACG/Assignment2/asm/asm	-t	4	-k	31	

System information:
  SPAdes version: 4.3.0
  Python version: 3.14.7
  OS: macOS-26.4.1-arm64-arm-64bit-Mach-O

Output dir: /Users/ashrithkollu/Documents/Classes/Junior_Fall/CG-ACG/Assignment2/asm/asm
Mode: ONLY assembling (without read error correction)
Debug mode is turned OFF

Dataset parameters:
  Isolate mode
  Reads:
    L

CompletedProcess(args=['spades.py', '--isolate', '--pe1-1', 'frag180.1.fq', '--pe1-2', 'frag180.2.fq', '--mp1-1', 'jump2k.1.fq', '--mp1-2', 'jump2k.2.fq', '-o', 'asm', '-t', '4', '-k', '31'], returncode=0)

In [7]:
os.chdir('/Users/ashrithkollu/Documents/Classes/Junior_Fall/CG-ACG/Assignment2/asm/asm')
result = subprocess.run(['grep', '-c', '>', 'contigs.fasta'], capture_output=True, text=True)

print(f"{result.stdout.strip()} contigs were produced")

4 contigs were produced


Question 4b. What is the total length of the contigs?

In [12]:
subprocess.run(['samtools', 'faidx', 'contigs.fasta'], check=True)

length_array = []
with open("contigs.fasta.fai") as file:
    for line in file:
        contig, length, offset, line_bases, line_width = line.strip().split("\t")
        length_array.append(int(length))

print(f"Total contig length is {sum(length_array)}")

Total contig length is 234569


Question 4c. What is the size of your largest contig?

In [14]:
length_array.sort(reverse=True)
print(f"The largest contig is {length_array[0]} bp long")

The largest contig is 105834 bp long


Question 4d. What is the contig N50 size? 


In [19]:
def find_N50(length_array):

    n50_sum = 0

    for value in length_array:
        n50_sum += value

        if n50_sum >= 1.0/2.0*sum(length_array):
            return value
    return 0

print(f"The N50 size is {find_N50(length_array)}")

The N50 size is 47849


#### **Question 5. Finding and decoding the insertion**

Use the `mummer`software tools to align your contigs to the reference genome. This will reveal all of the structural variants present, including the secret insertion.

Question 5a. How many insertions and deletions are in the assembly?

In [21]:
result = subprocess.run(['dnadiff', '/Users/ashrithkollu/Documents/Classes/Junior_Fall/CG-ACG/Assignment2/asm/ref.fa', '/Users/ashrithkollu/Documents/Classes/Junior_Fall/CG-ACG/Assignment2/asm/asm/contigs.fasta'], capture_output=True, check=True)

print(result)

CompletedProcess(args=['dnadiff', '/Users/ashrithkollu/Documents/Classes/Junior_Fall/CG-ACG/Assignment2/asm/ref.fa', '/Users/ashrithkollu/Documents/Classes/Junior_Fall/CG-ACG/Assignment2/asm/asm/contigs.fasta'], returncode=0, stdout=b'', stderr=b'Building alignments\nFiltering alignments\nExtracting alignment coordinates\nAnalyzing SNPs\nExtracting alignment breakpoints\nGenerating report file\n')


Answer: There are 6 indels in the assembly. There are 5 deletions (present in reference and not in query) spanning 45 bases and 1 insertions (present in query but not reference) spanning 808 bases.

Question 5b. What is the position of the insertion on the reference?

In [23]:
#Generated by ChatGPT with the prompt just asking to turn show-coords table into something I can put into a notebook

import subprocess
import pandas as pd
from io import StringIO

result = subprocess.run(
    ["show-coords", "out.delta"],
    capture_output=True,
    text=True
)

df = pd.read_csv(
    StringIO(result.stdout),
    sep=r"\s+",
    skiprows=5,
    header=None
)

display(df)

,0,1,2,3,4,5,6,7,8,9,10,11,12
0,127960,233793,|,105834,1,|,105834,105834,|,100.00,|,Halomonas,NODE_1_length_105834_cov_18.177840
1,221067,221165,|,6995,6897,|,99,99,|,95.96,|,Halomonas,NODE_1_length_105834_cov_18.177840
2,226799,226897,|,12727,12629,|,99,99,|,95.96,|,Halomonas,NODE_1_length_105834_cov_18.177840
3,40664,88512,|,47849,1,|,47849,47849,|,100.00,|,Halomonas,NODE_2_length_47849_cov_17.929943
4,3,26789,|,41445,14659,|,26787,26787,|,100.00,|,Halomonas,NODE_3_length_41445_cov_18.207587
5,26790,40639,|,13850,1,|,13850,13850,|,100.00,|,Halomonas,NODE_3_length_41445_cov_18.207587
6,88516,127956,|,39441,1,|,39441,39441,|,100.00,|,Halomonas,NODE_4_length_39441_cov_18.559959


Answer: There's a gap between base 26789 and 26790 in the reference.

Question 5c. How long is the novel insertion?

Answer: At this position, there is an insertion between base pairs 13850 and 14659, implying an insertion of 808 base pairs. 

Question 5d. What is the DNA sequence of the encoded message?

In [25]:
import subprocess

contig = "NODE_3_length_41445_cov_18.207587"
start = 13851
end = 14658

result = subprocess.run(
    ["samtools", "faidx", "/Users/ashrithkollu/Documents/Classes/Junior_Fall/CG-ACG/Assignment2/asm/asm/contigs.fasta", f"{contig}:{start}-{end}"],
    capture_output=True,
    text=True,
    check=True
)

print(result.stdout)

>NODE_3_length_41445_cov_18.207587:13851-14658
TAAATATGTAAATATTTACCGAGTCCTGCCAGGCACTACTCTCGGACGAGTATAAGGGCA
TCCGCTTTGCAGGGGGGCGTTACCTACTCTAGTAAGAGATGCAGTCTTAACGACCTTCCT
GGTGTCTTCTATTAATGGACGACTGGATACATTGCGAACGAGTCTCCTTTACGAAGTGTG
TAGGGAGTCACGACCAGAATTACGTCCGGGGGTAGGACCTGAAGGCACGCAGATGATACG
AAGCTAAGAAACGCCGCCACTACGGTACGCATTGTGGATGTAACGCCTCCAATCCTTGTG
TAGTGTGGCACGCGATGACGATATTCCTAATCGGCTTTTGTAGTAGTTTATGCTGGTATT
CGGTGAGTCGGGTCTGAAGGCCATCAGTACCTATGTTAATGGAAGCCGCCGGTACTTGTG
GCTTCAGTACATACGGGACGATTATCCTAGCAGAAGCAAATCCTTAAATACGCGCGGCCG
ACCTTGCTGTTTGAGGGGAGTCCGATATTCAGCGTCTCATGGACTCCGTTGTACATTTGG
CCATAGTTTTCGTGGGTAGGAGATAGATGTTAGGCGGAGATTATGTTTTATTTAATCCTT
GAGTCAGTGTGGCCGGGCTTACGGTGGTTAGTAGCTGCCGGTGAGAATGTCTACATTTTT
GATTCCCCGCCTGGCTACATTTGTTCTGAAGTCCAGTAACTAAGACCCTCAGCGTAGCAT
GGATCAATCGTGTAAGGGCCGAAGCGATCCCTGTATAACTCTGTGAAGTAGTACATACAT
TCCTTAAATAATAACAGACTCCGGTGCT



Question 5e. What is the secret message? 

Congratulations to the 2026 JHU Applied Genomics class!!!! Keep on looking for little green aliens...




Commands ran:

\$ samtools faidx /Users/ashrithkollu/Documents/Classes/Junior_Fall/CG-ACG/Assignment2/asm/asm/contigs.fasta 'NODE_3_length_41445_cov_18.207587:13851-14658' > insertion.fa

\$ python3 dna-decode.py -d --input asm/insertion.fa

\$ python3 dna-decode.py -d --input asm/insertion.fa --rev_comp
